In [1]:
import pandas as pd
import numpy as np
import os
# Save df_final as a .csv file
os.chdir(r'D:\sample_dataset')
df_final_cleaned=pd.read_csv('df_final_cleaned.csv', low_memory=False)

In [2]:
df_final_cleaned.shape

(1500000, 22)

In [3]:
# Separate the numerical and categorical features
numerical_features = df_final_cleaned.select_dtypes(include=['int64', 'float64']).columns
categorical_features = df_final_cleaned.select_dtypes(include=['object']).columns

# Count the number of numerical and categorical features
num_numerical = len(numerical_features)
num_categorical = len(categorical_features)

# Display the counts
print(f'Number of numerical features: {num_numerical}')
print(f'Number of categorical features: {num_categorical}')


Number of numerical features: 5
Number of categorical features: 17


In [4]:
# Count the number of attack and normal instances in the 'Label' column
label_counts = df_final_cleaned['Label'].value_counts()

# Display the result
print(label_counts)

Normal              1372539
http-flood            62383
http-loris            17502
quic-flood            17159
http2-concurrent       8980
fuzzing                8046
http2-pause            7458
quic-loris             3762
quic-enc               1663
http-smuggle            508
Name: Label, dtype: int64


In [5]:
df=df_final_cleaned.copy()

In [6]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import LabelEncoder, StandardScaler
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.models import Model

In [7]:
# Fill missing values for categorical features
df.loc[:, categorical_features] = df[categorical_features].fillna(df[categorical_features].mode().iloc[0])

In [8]:
# Fill missing values for numerical features
df.loc[:, numerical_features] = df[numerical_features].fillna(df[numerical_features].mean())

In [9]:
# Check the data types of the columns
print(df[categorical_features].dtypes)

frame.time            object
eth.dst               object
eth.src               object
ip.hdr_len            object
ip.dsfield.dscp       object
ip.dsfield.ecn        object
ip.len                object
ip.checksum           object
ip.checksum.status    object
ip.flags.rb           object
ip.flags.df           object
ip.flags.mf           object
ip.ttl                object
ip.src                object
ip.id                 object
ip.dst                object
Label                 object
dtype: object


In [10]:
# Convert categorical columns to string type for Label Encoding
for col in categorical_features:
    df[col] = df[col].astype(str)


In [11]:
# Apply Label Encoding to categorical features
label_encoder = LabelEncoder()
for col in categorical_features:
    df[col] = label_encoder.fit_transform(df[col])

In [12]:
# Normalize the entire dataset (including numerical features)
scaler = StandardScaler()
df[numerical_features] = scaler.fit_transform(df[numerical_features])


In [13]:
# Identify and extract the minority classes
minority_classes = df[df['Label'].isin([7, 4])]  # '7' and '4' are for 'quic-enc' and 'http-smuggle'
print("Shape of minority_classes after filtering:", minority_classes.shape)


Shape of minority_classes after filtering: (2171, 22)


In [20]:
minority_classes.head()

,frame.time,frame.time_epoch,frame.time_delta,frame.time_delta_displayed,frame.time_relative,frame.len,eth.dst,eth.src,ip.hdr_len,ip.dsfield.dscp,...,ip.checksum,ip.checksum.status,ip.flags.rb,ip.flags.df,ip.flags.mf,ip.ttl,ip.src,ip.id,ip.dst,Label
900070,722001,-0.703220,-0.319018,-0.319018,-0.013959,-0.443343,2,6,0,0,...,48623,0,0,0,0,21,20,1,0,7
900086,719598,-0.703278,-0.319018,-0.319018,-0.017712,-0.433561,2,6,0,0,...,48591,0,0,0,0,21,20,1,0,7
900204,716943,-0.703356,-0.319018,-0.319018,-0.022716,-0.442454,2,6,0,0,...,48611,0,0,0,0,21,20,1,0,7
901383,718035,-0.703309,-0.319018,-0.319018,-0.019710,-0.443343,2,6,0,0,...,48623,0,0,0,0,21,20,1,0,7
901448,718930,-0.703293,-0.319018,-0.319018,-0.018660,-0.433561,2,6,0,0,...,48591,0,0,0,0,21,20,1,0,7


In [19]:
df.head()

,frame.time,frame.time_epoch,frame.time_delta,frame.time_delta_displayed,frame.time_relative,frame.len,eth.dst,eth.src,ip.hdr_len,ip.dsfield.dscp,...,ip.checksum,ip.checksum.status,ip.flags.rb,ip.flags.df,ip.flags.mf,ip.ttl,ip.src,ip.id,ip.dst,Label
0,29900,-0.738911,-0.319018,-0.319018,-0.031409,0.105372,6,2,0,0,...,26900,0,0,2,0,22,0,0,17,0
1,29869,-0.738913,-0.319018,-0.319018,-0.031483,-0.436673,2,6,0,2,...,2927,0,0,2,0,21,20,29312,0,0
2,97400,-0.737475,-0.319018,-0.319018,0.060695,0.105372,6,2,0,0,...,26900,0,0,2,0,22,0,0,17,0
3,40239,-0.738701,-0.319018,-0.319018,-0.017918,0.105372,6,2,0,0,...,26900,0,0,2,0,22,0,0,17,2
4,14175,-0.739548,-0.319018,-0.319018,-0.072260,0.105372,6,2,0,0,...,26900,0,0,2,0,22,0,0,17,0


In [23]:
# Count the number of attack and normal instances in the 'Label' column
label_counts = df['Label'].value_counts()

# Display the result
print(label_counts)

0    1372539
2      62383
3      17502
8      17159
5       8980
1       8046
6       7458
9       3762
7       1663
4        508
Name: Label, dtype: int64


In [21]:
# Define the generator model
def build_generator(latent_dim, num_features):
    model = tf.keras.Sequential()
    model.add(layers.Input(shape=(latent_dim,)))  # Use Input layer instead of input_dim
    model.add(layers.Dense(128))
    model.add(layers.LeakyReLU(negative_slope=0.2))  # Changed alpha to negative_slope as recommended
    model.add(layers.BatchNormalization(momentum=0.8))
    model.add(layers.Dense(256))
    model.add(layers.LeakyReLU(negative_slope=0.2))
    model.add(layers.BatchNormalization(momentum=0.8))
    model.add(layers.Dense(num_features, activation='tanh'))  # Output as normalized data
    return model

# Define the critic (discriminator) model
def build_critic(num_features):
    model = tf.keras.Sequential()
    model.add(layers.Input(shape=(num_features,)))  # Use Input layer instead of input_dim
    model.add(layers.Dense(256))
    model.add(layers.LeakyReLU(negative_slope=0.2))
    model.add(layers.Dense(128))
    model.add(layers.LeakyReLU(negative_slope=0.2))
    model.add(layers.Dense(1))  # Single output to indicate real or fake
    return model

# Define the WGAN-GP class
class WGANGP(Model):
    def __init__(self, generator, critic, latent_dim, num_features):
        super(WGANGP, self).__init__()
        self.generator = generator
        self.critic = critic
        self.latent_dim = latent_dim
        self.num_features = num_features

    def compile(self, g_optimizer, d_optimizer, loss_fn):
        super(WGANGP, self).compile()
        self.g_optimizer = g_optimizer
        self.d_optimizer = d_optimizer
        self.loss_fn = loss_fn

    def gradient_penalty(self, real_samples, fake_samples):
        batch_size = tf.shape(real_samples)[0]  # Ensure batch size is properly defined
        alpha = tf.random.normal([batch_size, 1])
        interpolated = alpha * real_samples + (1 - alpha) * fake_samples
        with tf.GradientTape() as gp_tape:
            gp_tape.watch(interpolated)
            pred = self.critic(interpolated)
        grads = gp_tape.gradient(pred, [interpolated])[0]
        norm = tf.sqrt(tf.reduce_sum(tf.square(grads), axis=[1]))
        gp = tf.reduce_mean((norm - 1.0) ** 2)
        return gp

    def train_step(self, real_samples):
        batch_size = tf.shape(real_samples)[0]
        # Train critic
        for _ in range(5):
            random_latent_vectors = tf.random.normal(shape=(batch_size, self.latent_dim))
            with tf.GradientTape() as tape:
                fake_samples = self.generator(random_latent_vectors)
                fake_logits = self.critic(fake_samples)
                real_logits = self.critic(real_samples)
                d_cost = tf.reduce_mean(fake_logits) - tf.reduce_mean(real_logits)
                gp = self.gradient_penalty(real_samples, fake_samples)
                d_loss = d_cost + gp * 10.0
            grads = tape.gradient(d_loss, self.critic.trainable_weights)
            self.d_optimizer.apply_gradients(zip(grads, self.critic.trainable_weights))

        # Train generator
        random_latent_vectors = tf.random.normal(shape=(batch_size, self.latent_dim))
        with tf.GradientTape() as tape:
            fake_samples = self.generator(random_latent_vectors)
            fake_logits = self.critic(fake_samples)
            g_loss = -tf.reduce_mean(fake_logits)
        grads = tape.gradient(g_loss, self.generator.trainable_weights)
        self.g_optimizer.apply_gradients(zip(grads, self.generator.trainable_weights))

        return {"d_loss": d_loss, "g_loss": g_loss}

In [22]:
# Set parameters for WGAN-GP
latent_dim = 100  # Latent space dimension for random input to generator
num_features = df.shape[1] - 1  # Number of features excluding the 'Label' column

# Build the generator and critic models
generator = build_generator(latent_dim, num_features)
critic = build_critic(num_features)

# Instantiate the WGAN-GP model
wgan = WGANGP(generator, critic, latent_dim, num_features)

# Compile the WGAN-GP model
wgan.compile(
    g_optimizer=tf.keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5, beta_2=0.9),
    d_optimizer=tf.keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5, beta_2=0.9),
    loss_fn=tf.keras.losses.BinaryCrossentropy(from_logits=True)
)


In [24]:
# Identify and extract the minority classes
minority_classes = df[df['Label'].isin([7])]  # Adjust to match your minority class labels
minority_data1 = minority_classes.drop(columns=['Label']).values.astype('float32')


In [25]:
# Train the WGAN-GP model
wgan.fit(minority_data1, batch_size=64, epochs=1000)


Epoch 1/1000
26/26 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - d_loss: -1571590.0000 - g_loss: -2.6292
Epoch 2/1000
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - d_loss: -6904260.5000 - g_loss: -20.3586
Epoch 3/1000
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - d_loss: -16574790.0000 - g_loss: -56.0276
Epoch 4/1000
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - d_loss: -31670692.0000 - g_loss: -114.4661
Epoch 5/1000
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - d_loss: -53274700.0000 - g_loss: -202.0373
Epoch 6/1000
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - d_loss: -82419064.0000 - g_loss: -322.4785
Epoch 7/1000
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - d_loss: -120205336.0000 - g_loss: -476.3764
Epoch 8/1000
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - d_loss: -167558960.0000 - g_loss: -670.1791
Epoch 9/1000
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - d_loss: -225479312.0000 - g_loss: -907.0682
Epoch 10/1000
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - d_loss: -294929344.0000 - g_loss: -1184.0190
Epoch 11/1000
26/26 ━━━━━━━━━━

In [26]:
# Generate synthetic samples for minority classes
synthetic_data1 = wgan.generator.predict(tf.random.normal(shape=(2000, latent_dim)))  # Generate synthetic samples


63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


In [27]:
# Identify and extract the minority classes
minority_classes2 = df[df['Label'].isin([4])]  # Adjust to match your minority class labels
minority_data2 = minority_classes2.drop(columns=['Label']).values.astype('float32')

In [28]:
# Train the WGAN-GP model
wgan.fit(minority_data2, batch_size=64, epochs=1000)


Epoch 1/1000
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - d_loss: -46012825600.0000 - g_loss: -61610.3125 
Epoch 2/1000
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - d_loss: -47564644352.0000 - g_loss: -62316.8398 
Epoch 3/1000
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - d_loss: -45743386624.0000 - g_loss: -62998.6992 
Epoch 4/1000
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - d_loss: -46079254528.0000 - g_loss: -63684.2266 
Epoch 5/1000
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - d_loss: -46056603648.0000 - g_loss: -64475.6406 
Epoch 6/1000
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - d_loss: -47835332608.0000 - g_loss: -65186.0312 
Epoch 7/1000
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - d_loss: -46493306880.0000 - g_loss: -65882.2812 
Epoch 8/1000
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - d_loss: -47142158336.0000 - g_loss: -66708.4922 
Epoch 9/1000
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - d_loss: -47728037888.0000 - g_loss: -67402.8984 
Epoch 10/1000
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - d_loss: -46467584000.0000 - g_loss: -6

In [29]:
# Generate synthetic samples for minority classes
synthetic_data2 = wgan.generator.predict(tf.random.normal(shape=(2000, latent_dim)))  # Generate synthetic samples

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 791us/step


In [30]:
# Convert synthetic data to DataFrame and assign labels
synthetic_df1 = pd.DataFrame(synthetic_data1, columns=df.columns[:-1])  # Exclude the 'Label' column
synthetic_df1['Label'] = 7  # Assign one of the minority class labels, adjust as needed
synthetic_df2 = pd.DataFrame(synthetic_data2, columns=df.columns[:-1])  # Exclude the 'Label' column
synthetic_df2['Label'] = 4  # Assign one of the minority class labels, adjust as needed


In [31]:
# Combine synthetic data with the original dataset
df_final_with_synthetic = pd.concat([df, synthetic_df1,synthetic_df2], ignore_index=True)


In [32]:
df_final_with_synthetic.shape

(1504000, 22)

In [ ]:
os.chdir(r'D:\sample_dataset')
# Save the final dataframe as a CSV file
df_final_with_synthetic.to_csv('augmented_dataset.csv', index=False)